# Data Leakage Analysis

**CRITICAL ISSUE IDENTIFIED:** The Sports Reference stats include tournament games!

This means teams that go deeper in the tournament have:
- More wins → higher win%
- More games → potentially different stats
- Inflated SRS from beating tournament opponents

**This is data leakage** - the model "sees" tournament outcomes in the features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

## 1. Quantify the Leakage

In [ ]:
# Load 2024 data
stats = pd.read_csv('../data/raw/sportsref_combined_2024.csv')
tourney = pd.read_csv('../data/raw/espn_tournament_2024.csv')

# Count tournament games per team
home_games = tourney.groupby('home_team_name').size()
away_games = tourney.groupby('away_team_name').size()
tourney_games = pd.concat([home_games, away_games]).groupby(level=0).sum()

print(f'Teams in tournament: {len(tourney_games)}')
print(f'\nTournament games by finish:')
print(f'  Champion: 6 games')
print(f'  Runner-up: 6 games')
print(f'  Final Four: 5 games')
print(f'  Elite 8: 4 games')
print(f'  Sweet 16: 3 games')
print(f'  Round of 32: 2 games')
print(f'  Round of 64: 1 game')

In [ ]:
# Show teams with most games (deepest runs)
print('Teams with 37+ games (tournament participants):')
tournament_teams = stats[stats['Overall_G'] >= 37][['School', 'Overall_G', 'Overall_W', 'Overall_SRS']]
tournament_teams = tournament_teams.sort_values('Overall_G', ascending=False)
print(tournament_teams.head(15).to_string(index=False))

## 2. Impact on Key Features

In [ ]:
# Compare champion vs average team
uconn = stats[stats['School'] == 'Connecticut'].iloc[0]
avg_team = stats.mean(numeric_only=True)

print('UConn (Champion) vs Average Team:')
print(f"  Games: {uconn['Overall_G']:.0f} vs {avg_team['Overall_G']:.1f} (avg)")
print(f"  Win%: {uconn['Overall_W-L%']:.3f} vs {avg_team['Overall_W-L%']:.3f} (avg)")
print(f"  SRS: {uconn['Overall_SRS']:.2f} vs {avg_team['Overall_SRS']:.2f} (avg)")

# The extra 6 tournament wins inflate UConn's stats significantly

In [ ]:
# Estimate pre-tournament stats for UConn
# They played 40 games total, 6 were tournament
# So ~34 pre-tournament games

pre_tourney_games = 34
pre_tourney_wins = uconn['Overall_W'] - 6  # Subtract 6 tournament wins
pre_tourney_win_pct = pre_tourney_wins / pre_tourney_games

print(f'\nEstimated PRE-tournament UConn stats:')
print(f'  Games: {pre_tourney_games}')
print(f'  Wins: {pre_tourney_wins:.0f}')
print(f'  Win%: {pre_tourney_win_pct:.3f} (vs {uconn["Overall_W-L%"]:.3f} with tournament)')
print(f'\n  Inflation from tournament: +{(uconn["Overall_W-L%"] - pre_tourney_win_pct)*100:.1f}% win rate')

## 3. How This Affects Predictions

When we predict UConn vs Purdue in the final:
- UConn's stats include their 5 wins TO GET to the final
- Purdue's stats include their 5 wins TO GET to the final
- The model sees inflated stats for both teams

**Why this creates artificial accuracy:**
- Teams with good tournament runs have better stats
- Model predicts teams with better stats to win
- Those teams DID win (that's why their stats are inflated)
- Self-fulfilling prophecy!

In [ ]:
# Visualize: Tournament depth vs Game count
fig, ax = plt.subplots(figsize=(10, 6))

# All teams
ax.hist(stats['Overall_G'], bins=range(25, 45), alpha=0.5, label='All Teams', color='blue')

# Mark tournament teams
ax.axvline(x=37, color='red', linestyle='--', label='Typical tournament cutoff')
ax.axvline(x=40, color='green', linestyle='--', label='Champion (40 games)')

ax.set_xlabel('Total Games Played')
ax.set_ylabel('Number of Teams')
ax.set_title('Game Count Distribution - 2024 Season\n(Shows tournament games inflating stats)')
ax.legend()
plt.tight_layout()
plt.savefig('../results/visualizations/data_leakage_games.png', dpi=150)
plt.show()

## 4. Severity of the Problem

This leakage affects:
1. **Win percentage** - Direct leakage (tournament wins counted)
2. **SRS** - Inflated by beating tournament-quality opponents
3. **Points for/against** - More data from tournament games
4. **All counting stats** - Tournament games included

The deeper a team goes, the MORE their stats are inflated.

In [ ]:
# Calculate correlation between games played and tournament success
# This shouldn't exist in pre-tournament data!

# Map team names to tournament games
name_mapping = {
    'Connecticut': 'UConn Huskies',
    'Purdue': 'Purdue Boilermakers',
    'North Carolina State': 'NC State Wolfpack',
    'Alabama': 'Alabama Crimson Tide',
}

# For teams in tournament, games above 34 ≈ tournament games
stats['est_tourney_games'] = (stats['Overall_G'] - 34).clip(lower=0)

# Correlation with SRS
corr = stats[stats['est_tourney_games'] > 0][['est_tourney_games', 'Overall_SRS']].corr().iloc[0, 1]
print(f'Correlation between tournament depth and SRS: {corr:.3f}')
print('(This is artificially high due to leakage!)')

## 5. Conclusion

**The model's 77% accuracy and 80% champion prediction rate are INVALID.**

The true accuracy is likely much lower because:
- Tournament outcomes are embedded in the features
- The model is essentially memorizing who won

**To fix this, we need:**
1. Pre-tournament snapshots of team stats
2. Or manual adjustment to subtract tournament games
3. For 2025 predictions, collect data BEFORE the tournament starts